In [ ]:
# - imports -

import os
import csv
from pathlib import Path
from rdflib import Graph, RDF, RDFS, SKOS, URIRef

In [ ]:

class MOSAIC:
    def __init__(self, lex_thres=0.8, sem_thres=0.7):
        # MOSAIC: Multi-strategy Ontology Alignment and Integration Suite at CSG
        # Initializes hyperparameters and model caches.
        self.lex_thres = lex_thres
        self.sem_thres = sem_thres
        self.embedding_model = None  # Cache for SentenceBERT (all-MiniLM-L6-v2)

    def load_ontology(self, file_path: Path) -> Graph:
        # Loads an ontology file (.rdf, .owl, or .ttl) into an RDFlib Graph.
        g = Graph()
        # Handle xml parsing format for both standard .rdf and .owl extensions
        file_format = "turtle" if file_path.suffix == ".ttl" else "xml"
        try:
            g.parse(str(file_path), format=file_format)
            return g
        except Exception as e:
            print(f" [MOSAIC] Error loading {file_path.name}: {e}")
            return None

    def extract_entities(self, graph: Graph):
        # Extracts classes and properties along with their lexical labels.
        entities = {}
        # Query for named Classes
        for s in graph.subjects(RDF.type, URIRef("http://www.w3.org/2002/07/owl#Class")):
            if isinstance(s, URIRef):
                # Fallback to fragment/local name if no rdfs:label exists
                label = graph.value(s, RDFS.label) or graph.value(s, SKOS.prefLabel) or s.split("#")[-1]
                entities[s] = str(label).lower()
        return entities

    def lexical_blocking(self, source_entities, target_entities):
        # Inverted Index / Token Blocking (AdvancedAlign)
        # Implement candidate pairs selection to drastically reduce O(N*M) space
        candidate_pairs = []
        return candidate_pairs

    def semantic_similarity(self, candidate_pairs):
        # SentenceBERT Vector Scoring (PairMap).
        # Implement encoding and cosine similarity tracking
        return []

    def extract_final_alignments(self, scored_pairs):
        # Greedy 1-to-1 Mapping Extraction.
        # Filter competing matches to maximize precision
        return []

    def align(self, source_graph: Graph, target_graph: Graph):
        # Main pipeline orchestration for MOSAIC.
        src_entities = self.extract_entities(source_graph)
        tgt_entities = self.extract_entities(target_graph)
        
        # Implement actual alignment logic components.
        # For evaluation testing, this returns a placeholder match list matching identical names.
        final_alignments = set()
        for s_uri, s_lbl in src_entities.items():
            for t_uri, t_lbl in tgt_entities.items():
                if s_lbl == t_lbl and s_lbl != "":
                    final_alignments.add((str(s_uri), str(t_uri)))
                    
        return final_alignments


class OAEITrackRunner:
    def __init__(self, matcher: MOSAIC):
        # Handles directory parsing and pipeline routing for OAEI tracks.
        self.matcher = matcher
        self.results_log = []  # Holds statistics for final file output

    def load_reference_alignments(self, reference_path: Path) -> set:
        # Parses the ground-truth mappings from the task .ttl file.
        ref_set = set()
        ref_graph = Graph()
        # Define the equivalentClass predicate used to bind OAEI evaluation entities
        equivalent_class_predicate = URIRef("http://www.w3.org/2002/07/owl#equivalentClass")
        try:
            ref_graph.parse(str(reference_path), format="turtle")
            # Pull equivalent mappings from the reference graph matching (target, predicate, source) format
            for s, p, o in ref_graph.triples((None, equivalent_class_predicate, None)):
                ref_set.add((str(o), str(s)))
        except Exception as e:
            print(f" Could not read reference file {reference_path.name}: {e}")
        return ref_set

    def calculate_metrics(self, system_alignments: set, reference_alignments: set):
        # Calculates Precision, Recall, and F1-score.
        if not reference_alignments:
            return 0.0, 0.0, 0.0
            
        true_positives = len(system_alignments.intersection(reference_alignments))
        
        precision = true_positives / len(system_alignments) if len(system_alignments) > 0 else 0.0
        recall = true_positives / len(reference_alignments) if len(reference_alignments) > 0 else 0.0
        f_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        return round(precision, 4), round(recall, 4), round(f_score, 4)

    def run_all_tracks(self, tracks_base_dir: str, output_csv: str = "mosaic_evaluation_report.csv"):
        # Traverses the tracks folder to execute alignment tasks.
        base_path = Path(tracks_base_dir)
        
        if not base_path.exists():
            print(f"Error: Base directory '{tracks_base_dir}' does not exist.")
            return

        for track_folder in base_path.iterdir():
            if not track_folder.is_dir():
                continue
                
            print(f"\n" + "="*50)
            print(f" TRACK RUNNER: {track_folder.name.upper()}")
            print(f"="*50)
            
            alignment_tasks = list(track_folder.glob("*.ttl"))
            
            # Variables to calculate track averages
            track_p_sum, track_r_sum, track_f_sum = 0.0, 0.0, 0.0
            task_count = 0
            
            for task_file in alignment_tasks:
                parts = task_file.stem.split('-')
                if len(parts) != 2:
                    continue
                
                # Check for either .owl or .rdf extensions dynamically inside the ontologies subfolder
                source_path = track_folder / "ontologies" / f"{parts[0]}.owl"
                if not source_path.exists():
                    source_path = track_folder / "ontologies" / f"{parts[0]}.rdf"
                    
                # Check for either .owl or .rdf extensions dynamically inside the ontologies subfolder
                target_path = track_folder / "ontologies" / f"{parts[1]}.owl"
                if not target_path.exists():
                    target_path = track_folder / "ontologies" / f"{parts[1]}.rdf"
                
                print(f"\nMOSAIC Task: {parts[0]} ➔ {parts[1]}")
                
                if not source_path.exists() or not target_path.exists():
                    print(f" Skipping task. Missing source/target files.")
                    continue
                
                # Load ground truth reference and ontology graphs
                reference_alignments = self.load_reference_alignments(task_file)
                source_graph = self.matcher.load_ontology(source_path)
                target_graph = self.matcher.load_ontology(target_path)
                
                if source_graph and target_graph:
                    alignments = self.matcher.align(source_graph, target_graph)
                    print(f" Step complete. MOSAIC returned {len(alignments)} matches.")
                    
                    # Compute individual metrics
                    p, r, f1 = self.calculate_metrics(alignments, reference_alignments)
                    print(f"   Metrics -> Precision: {p}, Recall: {r}, F1-Score: {f1}")
                    
                    # Log task rows
                    self.results_log.append({
                        "Track": track_folder.name,
                        "Task": task_file.stem,
                        "Precision": p,
                        "Recall": r,
                        "F1-Score": f1,
                        "Type": "Task"
                    })
                    
                    track_p_sum += p
                    track_r_sum += r
                    track_f_sum += f1
                    task_count += 1
            
            # Compute track averages
            if task_count > 0:
                avg_p = round(track_p_sum / task_count, 4)
                avg_r = round(track_r_sum / task_count, 4)
                avg_f1 = round(track_f_sum / task_count, 4)
                
                print(f"\n Track [{track_folder.name}] AVERAGES -> Precision: {avg_p}, Recall: {avg_r}, F1-Score: {avg_f1}")
                
                self.results_log.append({
                    "Track": track_folder.name,
                    "Task": "TRACK_AVERAGE",
                    "Precision": avg_p,
                    "Recall": avg_r,
                    "F1-Score": avg_f1,
                    "Type": "Average"
                })

        # Save evaluations to file
        self.results_to_csv(output_csv)

    def results_to_csv(self, filename: str):
        # Outputs performance evaluation data directly to a CSV summary sheet.
        fields = ["Track", "Task", "Precision", "Recall", "F1-Score", "Type"]
        with open(filename, mode="w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fields)
            writer.writeheader()
            writer.writerows(self.results_log)
        print(f"\n Complete report successfully written to: {filename}")


if __name__ == "__main__":
    # Instantiate MOSAIC
    m = MOSAIC(lex_thres=0.85, sem_thres=0.75)
    
    # Pass into the track runner
    runner = OAEITrackRunner(matcher=m)
    runner.run_all_tracks("../tracks", output_csv="mosaic_evaluation_report.csv")